# Moduł 2: HTTP Protocol - Mechanika i Best Practices

**Ćwiczenie:** #2 - HTTP Mechanics

---

## Spis treści

1. [Wprowadzenie](#1-wprowadzenie)
2. [Kluczowe koncepcje](#2-kluczowe-koncepcje)
   - [HTTP Methods - Semantyka i Użycie](#21-http-methods---semantyka-i-użycie)
   - [HTTP Status Codes](#22-http-status-codes---komunikacja-wyniku)
   - [HTTP Headers](#23-http-headers---metadata-requestów-i-responseów)
   - [Content Negotiation](#24-content-negotiation-content-type-i-accept)
   - [CORS](#25-cors-cross-origin-resource-sharing)
3. [Best Practices](#3-best-practices)
4. [Demo Live Coding](#4-demo-live-coding)
5. [Przygotowanie do ćwiczenia](#5-przygotowanie-do-ćwiczenia)
6. [Dodatkowe materiały](#6-dodatkowe-materiały)

---

## 1. Wprowadzenie

### Po co ten temat?

HTTP to fundament komunikacji w web APIs. Zrozumienie:
- **HTTP methods** - kiedy użyć GET, POST, PUT, DELETE
- **Status codes** - jak komunikować wynik operacji
- **Headers** - jak przekazywać metadata (auth, content-type, CORS)

...jest kluczowe dla budowy **prawidłowych**, **zrozumiałych** i **zgodnych ze standardami** API.

### Gdzie to użyjemy w praktyce?

- **Status codes**: Informowanie klienta o sukcesie/błędzie (200, 404, 500)
- **Headers**: Authentication (Authorization), content negotiation (Accept)
- **Methods**: CRUD operations (GET=read, POST=create, PUT=update, DELETE=delete)
- **Custom headers**: Rate limiting, API versioning, correlation IDs

**Dlaczego to ważne?**
- API zgodne z konwencjami = łatwiejsze w użyciu
- Klienty (frontend, mobile) wiedzą jak interpretować odpowiedzi
- Standardy (RESTful) = uniwersalne zrozumienie

## 2. Kluczowe koncepcje

### 2.1. HTTP Methods - Semantyka i Użycie

#### GET - Pobierz zasoby (Read)

**Charakterystyka:**
- **Idempotentny** - wielokrotne wywołanie daje ten sam rezultat
- **Safe** - nie modyfikuje danych
- Parametry w URL (query string)
- Może być cache'owany

**Przykład:**

In [ ]:
@app.get("/users")
def get_users():
    """Pobierz listę wszystkich użytkowników"""
    return {"users": [...]}

@app.get("/users/{user_id}")
def get_user(user_id: int):
    """Pobierz konkretnego użytkownika"""
    return {"user_id": user_id, "name": "John"}

**Requesty:**
```bash
GET /users           → 200 OK, lista użytkowników
GET /users/42        → 200 OK, użytkownik 42
GET /users/999       → 404 Not Found (nie istnieje)
```

**Kiedy używać:**
- Pobieranie danych (bez modyfikacji)
- Lista zasobów
- Wyszukiwanie (search)
- Dashboard/raporty

#### POST - Stwórz nowy zasób (Create)

**Charakterystyka:**
- **NIE idempotentny** - wielokrotne wywołanie tworzy wiele zasobów
- **NIE safe** - modyfikuje dane
- Dane w body (JSON)
- **NIE** cache'owany

**Przykład:**

In [ ]:
@app.post("/users")
def create_user():
    """Stwórz nowego użytkownika"""
    # Body będzie parsowane automatycznie (następny moduł)
    return {"id": 123, "name": "John"}, 201

**Requesty:**
```bash
POST /users
Body: {"name": "John", "email": "john@example.com"}
→ 201 Created, zwraca utworzony obiekt
```

**Kiedy używać:**
- Tworzenie nowego zasobu
- Operacje bez oczywistego ID (np. search z body, upload)
- Akcje (actions), np. `/send-email`, `/calculate`

**Odpowiedź:**
- **201 Created** - zasób utworzony
- Opcjonalnie: header `Location: /users/123` (URL nowego zasobu)

#### PUT - Zaktualizuj cały zasób (Update/Replace)

**Charakterystyka:**
- **Idempotentny** - wielokrotne wywołanie daje ten sam rezultat
- **NIE safe** - modyfikuje dane
- Dane w body (JSON)
- Zastępuje **cały** zasób (complete replacement)

**Przykład:**

In [ ]:
@app.put("/users/{user_id}")
def update_user(user_id: int):
    """Zaktualizuj użytkownika (cały obiekt)"""
    # Body zawiera WSZYSTKIE pola
    return {"id": user_id, "name": "Updated"}

**Requesty:**
```bash
PUT /users/42
Body: {"name": "John Updated", "email": "new@example.com", "age": 30}
→ 200 OK (zaktualizowano)
→ 404 Not Found (użytkownik nie istnieje)
```

**Kiedy używać:**
- Aktualizacja całego zasobu (wszystkie pola)
- "Upsert" - update lub create jeśli nie istnieje (rzadziej)

**PUT vs PATCH:**
- **PUT**: Zastępuje cały zasób (musisz wysłać wszystkie pola)
- **PATCH**: Aktualizuje tylko wybrane pola (częściowa aktualizacja)

#### PATCH - Zaktualizuj częściowo (Partial Update)

**Charakterystyka:**
- **Idempotentny** (zazwyczaj)
- **NIE safe** - modyfikuje dane
- Dane w body (JSON) - tylko pola do zmiany

**Przykład:**

In [ ]:
@app.patch("/users/{user_id}")
def partial_update_user(user_id: int):
    """Zaktualizuj wybrane pola użytkownika"""
    # Body zawiera TYLKO pola do zmiany
    return {"id": user_id, "name": "Partially Updated"}

**Requesty:**
```bash
PATCH /users/42
Body: {"name": "New Name"}  # Tylko jedno pole!
→ 200 OK (zaktualizowano tylko name)
```

**Kiedy używać:**
- Aktualizacja pojedynczych pól (np. zmiana statusu)
- Partial updates (nie chcesz wysyłać całego obiektu)

#### DELETE - Usuń zasób

**Charakterystyka:**
- **Idempotentny** - wielokrotne wywołanie daje ten sam rezultat (zasób usunięty)
- **NIE safe** - modyfikuje dane
- Zazwyczaj bez body

**Przykład:**

In [ ]:
@app.delete("/users/{user_id}")
def delete_user(user_id: int):
    """Usuń użytkownika"""
    # Usuń z bazy danych
    return {"message": f"User {user_id} deleted"}

**Requesty:**
```bash
DELETE /users/42
→ 204 No Content (usunięto, brak body)
→ 404 Not Found (użytkownik nie istnieje)
→ 200 OK (usunięto, opcjonalnie zwróć dane)
```

**Kiedy używać:**
- Usuwanie zasobu
- Soft delete (zmiana statusu) - zazwyczaj PATCH lepsze

**Odpowiedź:**
- **204 No Content** - usunięto, brak treści odpowiedzi
- **200 OK** - usunięto, zwracasz dane usuniętego obiektu

### 2.2. HTTP Status Codes - Komunikacja wyniku

#### 2xx Success (Sukces)

| Kod | Nazwa | Kiedy używać |
|-----|-------|-------------|
| **200 OK** | Sukces | GET, PUT, PATCH - operacja zakończona sukcesem |
| **201 Created** | Utworzono | POST - zasób utworzony |
| **204 No Content** | Brak treści | DELETE - usunięto, brak body w odpowiedzi |

**Przykłady:**

In [ ]:
from fastapi import status

@app.get("/users/{user_id}", status_code=status.HTTP_200_OK)
def get_user(user_id: int):
    return {"user_id": user_id}

@app.post("/users", status_code=status.HTTP_201_CREATED)
def create_user():
    return {"id": 123, "name": "John"}

@app.delete("/users/{user_id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_user(user_id: int):
    # Usuń z bazy
    return  # Brak body dla 204!

#### 4xx Client Errors (Błędy klienta)

| Kod | Nazwa | Kiedy używać |
|-----|-------|-------------|
| **400 Bad Request** | Zły request | Błąd składni, nieprawidłowe dane |
| **401 Unauthorized** | Brak autoryzacji | Brak tokenu/credentiali (AUTH wymagane) |
| **403 Forbidden** | Brak uprawnień | Token OK, ale brak permissions |
| **404 Not Found** | Nie znaleziono | Zasób nie istnieje |
| **422 Unprocessable Entity** | Błąd walidacji | Validation error (FastAPI default) |

**Przykłady:**

In [ ]:
from fastapi import HTTPException, status

@app.get("/users/{user_id}")
def get_user(user_id: int):
    user = db.get_user(user_id)  # Mock
    if not user:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="User not found"
        )
    return user

**401 vs 403:**
- **401 Unauthorized**: "Kim jesteś?" (nie zalogowany, brak tokenu)
- **403 Forbidden**: "Wiem kim jesteś, ale nie masz dostępu" (token OK, brak permissions)

#### 5xx Server Errors (Błędy serwera)

| Kod | Nazwa | Kiedy używać |
|-----|-------|-------------|
| **500 Internal Server Error** | Błąd serwera | Nieobsłużony wyjątek, błąd aplikacji |
| **503 Service Unavailable** | Serwis niedostępny | Maintenance, przeciążenie |

**Przykłady:**

In [ ]:
@app.get("/users")
def get_users():
    try:
        users = db.query_users()  # Może rzucić exception
        return users
    except DatabaseError:
        raise HTTPException(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            detail="Database temporarily unavailable"
        )

**Zasada:**
- **FastAPI automatycznie** zwraca 500 dla nieobsłużonych wyjątków
- Lepiej obsłużyć konkretne błędy i zwrócić odpowiedni kod

### 2.3. HTTP Headers - Metadata requestów i response'ów

#### Request Headers (od klienta do serwera)

**Najpopularniejsze:**

| Header | Cel | Przykład |
|--------|-----|----------|
| `Authorization` | Uwierzytelnienie | `Bearer <token>` |
| `Content-Type` | Format body | `application/json` |
| `Accept` | Oczekiwany format odpowiedzi | `application/json` |
| `User-Agent` | Klient (browser, app) | `Mozilla/5.0...` |
| `X-Request-ID` | Correlation ID | `abc-123-def` |

**Przykład odczytu w FastAPI:**

In [ ]:
from fastapi import Header

@app.get("/users")
def get_users(
    authorization: str = Header(None),
    user_agent: str = Header(None)
):
    """Odczyt headers z requestu"""
    return {
        "auth": authorization,
        "user_agent": user_agent
    }

**Request:**
```bash
curl -H "Authorization: Bearer my-token" \
     -H "User-Agent: MyApp/1.0" \
     http://localhost:8000/users
```

**Uwaga:** FastAPI automatycznie konwertuje `User-Agent` → `user_agent` (snake_case)

#### Response Headers (od serwera do klienta)

**Najpopularniejsze:**

| Header | Cel | Przykład |
|--------|-----|----------|
| `Content-Type` | Format body | `application/json` |
| `Location` | URL nowo utworzonego zasobu | `/users/123` |
| `X-RateLimit-Remaining` | Rate limiting | `95` |
| `Access-Control-Allow-Origin` | CORS | `*` lub `https://example.com` |

**Przykład ustawienia w FastAPI:**

In [ ]:
from fastapi import Response

@app.post("/users")
def create_user(response: Response):
    """Ustawienie custom headers w odpowiedzi"""
    user_id = 123  # Mock
    response.headers["Location"] = f"/users/{user_id}"
    response.headers["X-Custom-Header"] = "custom-value"
    response.status_code = 201
    return {"id": user_id, "name": "John"}

**Response:**
```
HTTP/1.1 201 Created
Content-Type: application/json
Location: /users/123
X-Custom-Header: custom-value

{"id": 123, "name": "John"}
```

#### Custom Headers Best Practices

**1. Używaj prefixu `X-` dla custom headers:**

In [ ]:
response.headers["X-Request-ID"] = "abc-123"
response.headers["X-RateLimit-Remaining"] = "95"

**2. Authorization header (JWT):**

In [ ]:
from fastapi import Header, HTTPException

@app.get("/protected")
def protected_route(authorization: str = Header(...)):
    """Endpoint wymagający tokenu"""
    if not authorization.startswith("Bearer "):
        raise HTTPException(status_code=401, detail="Invalid token")

    token = authorization.split(" ")[1]
    # Weryfikuj token (później w module JWT)
    return {"message": "Authorized"}

### 2.4. Content Negotiation (Content-Type i Accept)

#### Content-Type (w request body)

**Określa format wysyłanych danych:**

```python
# Klient wysyła JSON
POST /users HTTP/1.1
Content-Type: application/json

{"name": "John", "email": "john@example.com"}
```

**FastAPI automatycznie:**
- Parsuje JSON jeśli `Content-Type: application/json`
- Parsuje form data jeśli `Content-Type: application/x-www-form-urlencoded`
- Parsuje multipart jeśli `Content-Type: multipart/form-data`

#### Accept (oczekiwany format odpowiedzi)

**Klient określa preferowany format:**

In [ ]:
from fastapi import Request
from fastapi.responses import JSONResponse, HTMLResponse

@app.get("/users")
def get_users(request: Request):
    """Zwróć JSON lub HTML w zależności od Accept header"""
    accept = request.headers.get("accept", "")

    users = [{"id": 1, "name": "John"}]

    if "text/html" in accept:
        html = "<ul><li>John</li></ul>"
        return HTMLResponse(content=html)

    return JSONResponse(content={"users": users})

**Requesty:**
```bash
# JSON (default)
curl -H "Accept: application/json" http://localhost:8000/users
→ {"users": [{"id": 1, "name": "John"}]}

# HTML
curl -H "Accept: text/html" http://localhost:8000/users
→ <ul><li>John</li></ul>
```

### 2.5. CORS (Cross-Origin Resource Sharing)

**Problem:**
Browser blokuje requesty z innej domeny (np. frontend na `http://localhost:3000` → API na `http://localhost:8000`)

**Rozwiązanie:** CORS headers

**Setup w FastAPI:**

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

# Konfiguracja CORS
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000"],  # Lista dozwolonych origin
    allow_credentials=True,
    allow_methods=["*"],  # Wszystkie metody (GET, POST, etc.)
    allow_headers=["*"],  # Wszystkie headers
)

@app.get("/users")
def get_users():
    return {"users": []}

**Efekt:**
```
HTTP/1.1 200 OK
Access-Control-Allow-Origin: http://localhost:3000
Access-Control-Allow-Credentials: true
```

**Uwaga:** W production używaj konkretnych origin, nie `["*"]` z `allow_credentials=True`!

## 3. Best Practices

### ✅ Rób tak:

**1. Używaj właściwych HTTP methods:**

In [ ]:
# ✅ Dobre - semantyka RESTful
@app.get("/users")        # Pobierz listę
@app.post("/users")       # Stwórz nowego
@app.put("/users/{id}")   # Zaktualizuj całość
@app.patch("/users/{id}") # Zaktualizuj częściowo
@app.delete("/users/{id}")# Usuń

**2. Zwracaj odpowiednie status codes:**

In [ ]:
# ✅ Dobre
@app.post("/users", status_code=status.HTTP_201_CREATED)
def create_user():
    return {"id": 123}

@app.delete("/users/{id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_user(id: int):
    return  # Brak body dla 204

**3. Używaj HTTPException dla błędów:**

In [ ]:
# ✅ Dobre
from fastapi import HTTPException

@app.get("/users/{id}")
def get_user(id: int):
    user = db.get(id)
    if not user:
        raise HTTPException(
            status_code=404,
            detail="User not found"
        )
    return user

**4. Dodawaj custom headers dla metadata:**

In [ ]:
# ✅ Dobre
from fastapi import Response

@app.post("/users")
def create_user(response: Response):
    user_id = 123
    response.headers["Location"] = f"/users/{user_id}"
    response.headers["X-Request-ID"] = "abc-123"
    return {"id": user_id}

**5. Obsługuj CORS dla frontend:**

In [ ]:
# ✅ Dobre
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://myapp.com"],
    allow_methods=["GET", "POST"],
    allow_headers=["*"]
)

### ❌ Nie rób tak:

**1. Nie używaj GET do modyfikacji danych:**

```python
# ❌ Złe - GET nie powinien modyfikować
@app.get("/users/{id}/delete")
def delete_user(id: int):
    db.delete(id)
    return {"message": "Deleted"}

# ✅ Dobre
@app.delete("/users/{id}")
def delete_user(id: int):
    db.delete(id)
    return {"message": "Deleted"}
```

**2. Nie zwracaj 200 dla wszystkiego:**

```python
# ❌ Złe
@app.post("/users")
def create_user():
    return {"id": 123}  # Default 200, powinno być 201!

# ✅ Dobre
@app.post("/users", status_code=201)
def create_user():
    return {"id": 123}
```

**3. Nie używaj custom status codes bez powodu:**

```python
# ❌ Złe - niestandardowy kod
@app.get("/users/{id}")
def get_user(id: int):
    if not exists(id):
        raise HTTPException(status_code=499, detail="Custom error")

# ✅ Dobre - standardowy kod
@app.get("/users/{id}")
def get_user(id: int):
    if not exists(id):
        raise HTTPException(status_code=404, detail="User not found")
```

**4. Nie zwracaj body dla 204 No Content:**

```python
# ❌ Złe - 204 nie powinno mieć body
@app.delete("/users/{id}", status_code=204)
def delete_user(id: int):
    return {"message": "Deleted"}  # Konflikt!

# ✅ Dobre - brak body
@app.delete("/users/{id}", status_code=204)
def delete_user(id: int):
    db.delete(id)
    return  # Brak body
```